# Demo: Retraction of RDFS Inference

This notebook extends [demo-rdfs-inference.ipynb](./demo-rdfs-inference.ipynb) from one-way materialization to update-aware materialization.
The goal is small but important: after an asserted fact is removed, consequences that depended on that fact should disappear as well, while unrelated schema consequences remain available.

That behavior is essential for agent-facing memory and tool state.
A Research Agent may revise a belief, retract a claim, or replace an extracted statement; the reasoning layer should not keep stale inferred facts alive merely because they were true in an earlier graph state.

## Where does this notebook fit?

| Topic | Notebook |
|-------|----------|
| RDFS inference + derivation inspection | [demo-rdfs-inference.ipynb](./demo-rdfs-inference.ipynb) |
| RDFS inference retraction after graph updates | **this notebook** |
| Contradiction diagnostics + explanation | [demo-contradiction-rules.ipynb](./demo-contradiction-rules.ipynb) |

---

## 1. Inference-Backed RDFLib Graph

We use an in-memory RDFLib store wrapped by `RETEStore` so that ordinary graph updates trigger RDFS materialization.
An `InMemoryDerivationLogger` is enabled as well so the setup mirrors the inference and contradiction demos, even though this notebook focuses on graph state rather than proof rendering.

> ℹ️: You don't need a `derivation_logger` to use RDFS inference.
> If you don't need explanations, just set it to `None` / omit it.

In [1]:
from rdflib import Dataset, Graph, URIRef
from rdflib.namespace import RDF, RDFS
from rdflib.plugins.stores.memory import Memory
from rdflib_reasoning.engine import (
    PRODUCTION_RDFS_RULES,
    InMemoryDerivationLogger,
    RETEEngineFactory,
)
from rdflib_reasoning.engine.rete_store import RETEStore

logger = InMemoryDerivationLogger()
factory = RETEEngineFactory(rules=PRODUCTION_RDFS_RULES, derivation_logger=logger)
store = RETEStore(Memory(), factory)
dataset = Dataset(store=store)
graph: Graph = dataset.default_graph

## 2. Triggering Inference

We create some basic terms and construct a minimal ontology of 3 statements.
Due to our forward-chaining inference engine, the production RDFS profile materializes a visible closure beyond those asserted triples.
In this run, the graph grows from 0 triples to 9 triples: the 3 asserted triples, the expected user-facing type entailments for `alice`, and schema-level consequences that follow from the subclass chain.

The important user experience is the same as in the first RDFS demo: using a graph like this should look like working with a normal RDFLib graph.
The reasoner is present, but ordinary RDFLib calls such as `graph.add(...)`, `len(graph)`, and Turtle serialization remain the surface area.

In [2]:
graph.bind("ex", "urn:example:")

alice = URIRef("urn:example:alice")
person = URIRef("urn:example:Person")
mammal = URIRef("urn:example:Mammal")
animal = URIRef("urn:example:Animal")

print("triples before assertions:", len(graph))

graph.add((alice, RDF.type, person))
graph.add((person, RDFS.subClassOf, mammal))
graph.add((mammal, RDFS.subClassOf, animal))

print("triples after assertions:", len(graph))
print("-" * 80)
print(graph.serialize(format="turtle"))

triples before assertions: 0
triples after assertions: 9
--------------------------------------------------------------------------------
@prefix ex: <urn:example:> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

ex:Animal a rdfs:Class .

ex:Mammal a rdfs:Class ;
    rdfs:subClassOf ex:Animal .

ex:Person a rdfs:Class ;
    rdfs:subClassOf ex:Animal,
        ex:Mammal .

ex:alice a ex:Animal,
        ex:Mammal,
        ex:Person .




## 3. Triggering Retraction

Now we remove only the asserted triple `(alice, rdf:type, Person)`.
That assertion was the source of the inferred facts that `alice` is also a `Mammal` and an `Animal` through `rdfs9`, so those dependent facts should be retracted from the materialized graph.

The class hierarchy itself should remain: `Person subClassOf Mammal`, `Mammal subClassOf Animal`, and the derived `Person subClassOf Animal` are still supported by the remaining schema assertions.
After retraction, the graph therefore contains 6 triples rather than returning all the way to an empty graph.

In [3]:
graph.remove((alice, RDF.type, person))

print("triples after retraction:", len(graph))
print("-" * 80)
print(graph.serialize(format="turtle"))

triples after retraction: 6
--------------------------------------------------------------------------------
@prefix ex: <urn:example:> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

ex:Animal a rdfs:Class .

ex:Mammal a rdfs:Class ;
    rdfs:subClassOf ex:Animal .

ex:Person a rdfs:Class ;
    rdfs:subClassOf ex:Animal,
        ex:Mammal .




## Notes

- Retraction is dependency-aware: the asserted type for `alice` is removed, and the inferred `alice` types that depended on it are removed with it.
- Schema consequences with independent support remain materialized after the update.
- This notebook complements [demo-contradiction-rules.ipynb](./demo-contradiction-rules.ipynb): contradictions show how the engine surfaces invalid states, while retraction shows how the engine recovers when a supporting claim is withdrawn.
- Agentic systems tie-in: mutable symbolic memory is a practical requirement for agent systems that need auditable, revisable beliefs rather than append-only text traces.